# USDA NASS Cropland Data Layer over the Ogallala aquifer on the 30 m Landsat/OpenET tiles: one CF file per tile and year

The Cropland Data Layer (CDL) is NASS's yearly 30 m map of crop types. This notebook takes NASS's national
files, places them on the same 30 m UTM tiles as this project's Landsat index and OpenET files, so that all
three line up pixel for pixel, and writes one small file per tile and year to Drive.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/USDA_NASS_CDL_30m_tiles/<year>/<tile>/USDA_NASS_CDL_30m_tiles_<tile>_<year>_Ogallala.nc` |
| Years | 2008 (the first year the CDL covers every aquifer state) → the latest year NASS has published |
| Layers | `cropland` (class code, with NASS's class names and colours), `confidence` (percent), `cultivated` |
| Grid | the 30 m UTM tiles of the Landsat and OpenET files (their `frames.json` is reused) |
| Pixels kept | pixels whose centre lies inside the aquifer polygon; the others hold 255 |

**Where each layer comes from** (every file says it, in `cropland_source`, `confidence_source`, `cultivated_source`)

| Layer | From NASS | From Earth Engine's copy (`USDA/NASS/CDL`) | Not available |
|---|---|---|---|
| `cropland` | 2008 → latest | – | – |
| `confidence` | 2017–2023 | 2008–2016 | 2024 on (NASS shows them only in its online viewer) |
| `cultivated` | 2025 | 2013–2023 | 2008–2012, 2024 |

From 2024 NASS makes the CDL at 10 m. As chosen, the notebook uses NASS's own 30 m copy of those years (NASS
made it by nearest neighbour from the 10 m map), so every year goes through the same chain. NASS's
cultivated layer of a release is built from the five most recent years.

**From NASS's grid to the tiles.** The CDL is on an Albers equal-area grid; the tiles are on UTM grids.
Each tile pixel takes the value of the CDL pixel that contains its centre (nearest neighbour, computed
exactly; GDAL's own warping was not used because its approximate transform picked a neighbouring pixel
for about 2 % of pixels in a test). No value is
averaged, mixed or filled: every value in the files is one of NASS's. Because the two grids are turned
against each other, a few CDL pixels are used twice and a few not at all along each row; this is the
unavoidable price of putting the CDL on the Landsat grid (keep NASS's own files if you need the original grid).

**How to run**

1. *Run all* with `TEST_MODE = True`: two 2048-pixel squares of two tiles for 2016 (NASS for crop types,
   Earth Engine for confidence and cultivated), into `.../_test/`. The 2016 national file (1.8 GB) is
   downloaded once.
2. Set `TEST_MODE = False` and *Run all*. Each year downloads NASS's national crop-type file (about 2 GB) and,
   for 2017–2023, its confidence file (about 6.5 GB); they are deleted once the year is done. Whenever the
   runtime dies or you stop it, run all again: finished files are skipped.

**What the notebook checks**

* A download must have the length the server states, and the GeoTIFF taken out of the zip must match the
  checksum stored in the zip.
* For every tile and layer, 2000 random pixels are compared with a lookup of the CDL pixel under their
  centre made by another library (GDAL, while the notebook itself uses pyproj). Every file is read back and compared before it is copied
  to Drive, with a checksum comparison.

**Notes**

* NASS: confidence values "should not be used as a measure of accuracy" (pixels can be right with low confidence).
* Class 0 is "Background" (outside NASS's mapped area); 255 is this notebook's fill.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# earthengine-api, numpy, geopandas, shapely, pyproj and requests are part of Colab.
!pip install -q rasterio netCDF4 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> short rehearsal into .../_test/ ; False -> every year from FIRST_YEAR
TEST_YEAR     = 2016       # year of the rehearsal (2016: crop types from NASS, confidence and cultivated from Earth Engine)
TEST_TILES    = 2          # tiles in the rehearsal
TEST_FRAME_PX = 2048       # None -> whole tiles; or a square of this many pixels of each tile

FIRST_YEAR = 2008          # first year processed (the first year the CDL covers every aquifer state)
LAST_YEAR  = None          # None -> the latest year NASS has published
ONLY_TILES = None          # None -> every tile; or e.g. ["z14e02n29"]

# ---- where each layer comes from ----
NASS_URL   = "https://www.nass.usda.gov/Research_and_Science/Cropland/Release/datasets/"
NASS_FILES = {             # layer -> {year: zip file at NASS_URL}; what NASS offers for download (checked 2026-10-14)
    "cropland":   {y: f"{y}_30m_cdls.zip" for y in range(2008, 2026)},   # 2024 on: NASS's own 30 m copy of its 10 m map
    "confidence": {**{y: f"{y}_30m_confidence_layer.zip" for y in range(2017, 2021)},
                   **{y: f"{y}_30m_Confidence_Layer.zip" for y in range(2021, 2024)}},
    "cultivated": {2025: "2025_cultivated_layer.zip"},
}
EE_FALLBACK = True         # layer-years NASS no longer offers are read from Earth Engine's copy of the CDL
                           # (USDA/NASS/CDL: confidence 2008-2023, cultivated 2013-2023); every file says which
                           # source each layer came from
PROJECT_ID = "msugw-503806"                                    # Earth Engine cloud project (for the fallback only)
EE_URL     = "https://earthengine-highvolume.googleapis.com"

# ---- the output grid: the 30 m tiles of the Landsat and OpenET files ----
TILE_PX     = 5120
FRAMES_FROM = ["/content/drive/MyDrive/MSUGWB/LANDSAT_C2L2_indices_30m_monthly_tiles_v2/frames.json",
               "/content/drive/MyDrive/MSUGWB/OPENET_v2_1_monthly_30m_tiles/frames.json",
               "/content/drive/MyDrive/MSUGWB/OPENET_v2_0_monthly_30m_tiles/frames.json"]
                           # the tile frames are copied from the first of these that exists, so every tile has
                           # the same name, x and y as in those products

COMPRESS_LEVEL = 2
CHUNK_YX       = 512
MIN_FREE_GB    = 25.0      # scratch space that must stay free (a confidence layer is 7 GB unpacked)
DOWNLOAD_TRIES = 5
KEEP_DOWNLOADS = False     # True -> keep NASS's zip files on scratch after a year is done
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, the layers ═══════════════
import re, gc, io, sys, json, glob, time, math, zlib, shutil, hashlib, logging, threading, warnings
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import shapely
import rasterio
import rasterio.features
import rasterio.warp
import rasterio.windows
from rasterio.enums import Resampling
from affine import Affine
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("cdl")
for _n in ("urllib3.connectionpool", "googleapiclient", "rasterio", "pyogrio"):
    logging.getLogger(_n).setLevel(logging.ERROR)

ROI_LABEL   = "Ogallala"
PRODUCT_TAG = "USDA_NASS_CDL_30m_tiles"
DRIVE_ROOT  = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT   = f"{DRIVE_ROOT}/{PRODUCT_TAG}" + ("/_test" if TEST_MODE else "")
SCRATCH     = "/content/cdl_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR     = f"{SCRATCH}/raw"                      # NASS's zip files and the GeoTIFFs taken out of them
NC_DIR      = f"{SCRATCH}/nc"
MASK_DIR    = f"{SCRATCH}/masks"
DISK_PATH   = "/content"
FRAMES_NAME = "frames.json"
LEDGER_NAME = "files_done.jsonl"
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR, MASK_DIR):
    os.makedirs(_d, exist_ok=True)

PIXEL = 30.0
TILE_M = TILE_PX * PIXEL
FILL = 255                                          # fill of every variable: outside the polygon or no source
LAYERS = ["cropland", "confidence", "cultivated"]
SOURCE_CODE = dict(none=0, nass=1, earth_engine=2)
LAYER_TEXT = dict(
    cropland="crop or land-cover class of the pixel (USDA NASS Cropland Data Layer)",
    confidence="NASS's confidence in the pixel's class, percent (not a measure of accuracy, according to NASS)",
    cultivated="cultivated / non-cultivated (NASS: cultivated when classed as a crop in at least two of five years)")


class PipelineStop(RuntimeError):
    """Clean stop: everything on Drive is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


class SceneUnavailable(Exception):
    pass


class RunStopping(Exception):
    pass


class RequestTooLarge(Exception):
    pass


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: aquifer polygon and the tiles (the frames of the Landsat / OpenET files) ═══════════════
_ROI_GDF = gpd.read_file(SHP_PATH)
if _ROI_GDF.crs is None:
    raise ValueError(f"{SHP_PATH} has no CRS")
_POLY = {}
TEST_MODE_FRAMES = TEST_MODE and TEST_FRAME_PX


def roi_polygon(crs):
    """The aquifer polygon in a given CRS (one copy per CRS). The shapefile's parts are merged once,
    in its own CRS; the merged outline is then reprojected, which is quick."""
    if None not in _POLY:
        g = _ROI_GDF.geometry
        u = g.union_all() if hasattr(g, "union_all") else g.unary_union
        _POLY[None] = u if u.is_valid else u.buffer(0)
    if crs not in _POLY:
        u = gpd.GeoSeries([_POLY[None]], crs=_ROI_GDF.crs).to_crs(crs).iloc[0]
        _POLY[crs] = u if u.is_valid else u.buffer(0)
    return _POLY[crs]

def tile_key(crs, i, j):
    return f"z{int(crs.split(':')[1]) % 100:02d}e{i:02d}n{j:02d}"

def _mask_path(key):
    return os.path.join(MASK_DIR, f"{key}.npy")

def _rasterize(F):
    return rasterio.features.geometry_mask([roi_polygon(F["crs"])], out_shape=(F["h"], F["w"]), invert=True,
                                           all_touched=False,          # pixel centre inside the polygon
                                           transform=Affine(PIXEL, 0.0, F["x0"], 0.0, -PIXEL, F["y0"]))

def _save_mask(key, mask):
    tmp = _mask_path(key) + ".part"
    with open(tmp, "wb") as f:
        np.save(f, np.packbits(mask))
    os.replace(tmp, _mask_path(key))

def frame_mask(key, F):
    """True where the pixel centre lies inside the aquifer polygon (kept on scratch, 1 bit per pixel)."""
    if os.path.exists(_mask_path(key)):
        bits = np.load(_mask_path(key))
        if bits.size * 8 >= F["h"] * F["w"]:
            mask = np.unpackbits(bits)[:F["h"] * F["w"]].reshape(F["h"], F["w"]).astype(bool)
            if int(mask.sum()) == F["n_roi"]:
                return mask
    mask = _rasterize(F)
    if int(mask.sum()) != F["n_roi"]:
        raise DataIntegrityError(f"{key}: the aquifer mask has {int(mask.sum())} pixels, its frame says {F['n_roi']}. "
                                 "The shapefile differs from the one the frames were made with.")
    _save_mask(key, mask)
    return mask

def make_frame(key, crs, ox, oy, i, j):
    """Frame of tile (i, j) of a zone whose lattice has its pixel edges at ox, oy (mod 30 m): the
    tile cut to the rectangle that bounds its aquifer pixels. None when it holds no aquifer pixel."""
    left, top = ox + i * TILE_M, oy + (j + 1) * TILE_M
    box = shapely.geometry.box(left, top - TILE_M, left + TILE_M, top)
    if not roi_polygon(crs).intersects(box):
        return None
    F = dict(crs=crs, x0=left, y0=top, w=TILE_PX, h=TILE_PX)
    mask = _rasterize(F)
    if not mask.any():
        return None
    rows, cols = np.flatnonzero(mask.any(axis=1)), np.flatnonzero(mask.any(axis=0))
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    if TEST_MODE and TEST_FRAME_PX:                       # a small square around the aquifer pixel nearest the centre
        rr, cc = np.nonzero(mask[r0:r1:8, c0:c1:8])
        if not rr.size:
            return None
        k = int(np.argmin((rr * 8 - (r1 - r0) / 2) ** 2 + (cc * 8 - (c1 - c0) / 2) ** 2))
        ra = max(r0, r0 + int(rr[k]) * 8 - TEST_FRAME_PX // 2)
        ca = max(c0, c0 + int(cc[k]) * 8 - TEST_FRAME_PX // 2)
        r0, r1, c0, c1 = ra, min(r1, ra + TEST_FRAME_PX), ca, min(c1, ca + TEST_FRAME_PX)
    sub = np.ascontiguousarray(mask[r0:r1, c0:c1])
    F = dict(crs=crs, x0=left + c0 * PIXEL, y0=top - r0 * PIXEL, w=c1 - c0, h=r1 - r0, n_roi=int(sub.sum()))
    _save_mask(key, sub)
    return F


def zone_tiles():
    """Tiles of every UTM zone the aquifer reaches, used only when no frames.json of the Landsat or
    OpenET files is found: the tiles of zone z that meet the part of the aquifer within 1.5 degrees of
    the zone's own longitudes. The Landsat files' tiles depend on where Landsat scenes lie and can
    differ a little at zone edges; their frames.json is therefore preferred."""
    poly = roi_polygon("EPSG:4326")
    cand = {}
    for zone in range(1, 61):
        lon0 = -180 + 6 * (zone - 1)
        band = shapely.geometry.box(lon0 - 1.5, -80, lon0 + 6 + 1.5, 84)
        part = poly.intersection(band)
        if part.is_empty:
            continue
        crs = f"EPSG:{32600 + zone}"
        part_utm = gpd.GeoSeries([part], crs="EPSG:4326").to_crs(crs).iloc[0]
        x0, y0, x1, y1 = part_utm.bounds
        ox = oy = 15.0                                   # Landsat's lattice: pixel edges at 15 m mod 30
        for i in range(math.floor((x0 - ox) / TILE_M), math.floor((x1 - ox) / TILE_M) + 1):
            for j in range(math.floor((y0 - oy) / TILE_M), math.floor((y1 - oy) / TILE_M) + 1):
                cand[tile_key(crs, i, j)] = (crs, ox, oy, i, j)
    return cand


def get_frames():
    """{tile: frame}. Taken from the first FRAMES_FROM file that exists (so the tiles are those of the
    Landsat / OpenET files), otherwise made here; stored in this product's folder either way."""
    path = os.path.join(DRIVE_OUT, FRAMES_NAME)
    if os.path.exists(path):
        with open(path) as f:
            doc = json.load(f)
        log.info("Tile frames from %s (%s)", path, doc.get("origin", "?"))
        return {k: v for k, v in doc["frames"].items() if v}
    frames, origin = None, None
    for src in FRAMES_FROM:
        if os.path.exists(src):
            with open(src) as f:
                doc = json.load(f)
            if doc.get("made_with", {}).get("tile_px") == TILE_PX and doc.get("made_with", {}).get("pixel") in (None, PIXEL):
                frames, origin = {k: v for k, v in doc["frames"].items() if v}, src
                break
    if frames is None:
        origin = "made here from the UTM zones (no frames.json of the Landsat or OpenET files was found)"
        frames = {}
        for key, args in sorted(zone_tiles().items()):
            F = make_frame(key, *args)
            if F:
                frames[key] = F
    if TEST_MODE_FRAMES:                       # the rehearsal: a square in the middle of each tile's frame
        small = {}
        for key, F in frames.items():
            n = min(TEST_FRAME_PX, F["w"], F["h"])
            c0, r0 = (F["w"] - n) // 2, (F["h"] - n) // 2
            G = dict(crs=F["crs"], x0=F["x0"] + c0 * PIXEL, y0=F["y0"] - r0 * PIXEL, w=n, h=n)
            m = _rasterize(G)
            if m.any():
                small[key] = dict(G, n_roi=int(m.sum()))
        frames = small
    local = os.path.join(NC_DIR, FRAMES_NAME)
    with open(local, "w") as f:
        json.dump(dict(origin=origin, made_with=dict(tile_px=TILE_PX, pixel=PIXEL, test_frame_px=TEST_FRAME_PX if TEST_MODE else None),
                       created=iso(utcnow()), frames=frames), f, indent=1)
    robust_drive_copy(local, path)
    os.remove(local)
    log.info("Tile frames: %d tiles, %s", len(frames), origin)
    return frames

In [ ]:
# ═══════════════ CELL 6: the sources: NASS's national GeoTIFFs, and Earth Engine's copy for what NASS no longer offers ═══════════════
HTTP = requests.Session()
HTTP.headers["User-Agent"] = "MSU groundwater project CDL notebook (python-requests)"
NET_WAITS_S = [30, 60, 120, 240, 300, 300, 300]     # pauses between attempts when NASS's server cannot be reached


def _use_ipv4_only():
    """'Network is unreachable' can mean the runtime has no IPv6 route while a host is tried over IPv6
    first; from then on only IPv4 addresses are used."""
    import urllib3.util.connection as uc
    if getattr(uc, "HAS_IPV6", False):
        uc.HAS_IPV6 = False
        log.warning("Network unreachable - using IPv4 only from now on")


def nass_head(name):
    """NASS's answer about one file (HTTP status and headers). A server that cannot be reached is asked
    again after growing pauses (about 20 minutes in all) and only then is the run stopped. A missing
    file (404) is an answer, not an error."""
    last = None
    for attempt, pause in enumerate([0] + NET_WAITS_S):
        if pause:
            log.warning("NASS's server could not be reached (%s) - trying again in %d s", str(last)[:120], pause)
            time.sleep(pause)
        try:
            r = HTTP.head(NASS_URL + name, timeout=60, allow_redirects=True)
            if r.status_code >= 500:
                raise IOError(f"HTTP {r.status_code}")
            return r
        except (IOError, requests.RequestException) as e:
            last = e
            if "unreachable" in str(e).lower():
                _use_ipv4_only()
    raise PipelineStop(f"NASS's server could not be reached for {name} ({str(last)[:200]}). Everything on Drive is safe "
                       "- run again later; finished files are skipped.")


def nass_latest_year():
    """The newest year for which NASS's crop-type file exists. Only a "not found" answer moves on to the
    year before; a server that cannot be reached stops the run instead of picking an older year."""
    for y in range(utcnow().year, 2007, -1):
        name = NASS_FILES["cropland"].get(y, f"{y}_30m_cdls.zip")
        r = nass_head(name)
        if r.status_code == 200 and int(r.headers.get("Content-Length", 0)) > 10 ** 8:
            NASS_FILES["cropland"].setdefault(y, name)
            return y
    raise PipelineStop("No CDL file could be found at NASS.")


def download_nass(name):
    """A NASS zip file on scratch, complete: streamed in one request and continued with byte ranges after
    a break; its length must equal the server's. Returns the local path."""
    url, path = NASS_URL + name, os.path.join(RAW_DIR, name)
    head = nass_head(name)
    if head.status_code != 200:
        raise PipelineStop(f"NASS does not offer {name} (HTTP {head.status_code}).")
    size = int(head.headers["Content-Length"])
    if os.path.exists(path) and os.path.getsize(path) == size:
        return path
    ensure_disk_free(f"download {name}", extra_gb=size / 1024 ** 3)
    part, t0 = path + ".part", time.time()
    for attempt in range(1, DOWNLOAD_TRIES + 1):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        try:
            hdr = {"Range": f"bytes={have}-"} if have else {}
            with HTTP.get(url, headers=hdr, stream=True, timeout=120) as r:
                if have and r.status_code != 206:
                    have = 0                     # the server ignored the range: start again
                elif r.status_code not in (200, 206):
                    raise IOError(f"HTTP {r.status_code}")
                with open(part, "ab" if have else "wb") as f:
                    for chunk in r.iter_content(8 * 1024 * 1024):
                        f.write(chunk)
            if os.path.getsize(part) == size:
                os.replace(part, path)
                TOTALS["bytes"] = TOTALS.get("bytes", 0) + size
                log.info("Downloaded %s: %s in %.0f s", name, human(size), time.time() - t0)
                return path
            raise IOError(f"{os.path.getsize(part)} of {size} bytes")
        except (IOError, requests.RequestException) as e:
            log.warning("Download of %s broke off (%s), attempt %d of %d", name, str(e)[:120], attempt, DOWNLOAD_TRIES)
            if "unreachable" in str(e).lower():
                _use_ipv4_only()
            time.sleep(NET_WAITS_S[min(attempt, len(NET_WAITS_S)) - 1])
    raise PipelineStop(f"{name} could not be downloaded completely. Everything on Drive is safe - run again.")


def nass_tif(zip_path):
    """The GeoTIFF inside a NASS zip, taken out to scratch. zipfile checks it against the CRC the zip
    stores, so a damaged download cannot pass."""
    with zipfile.ZipFile(zip_path) as z:
        tifs = [i for i in z.infolist() if i.filename.lower().endswith(".tif")]
        if len(tifs) != 1:
            raise PipelineStop(f"{os.path.basename(zip_path)} holds {len(tifs)} GeoTIFFs; one was expected.")
        out = os.path.join(RAW_DIR, tifs[0].filename)
        if os.path.exists(out) and os.path.getsize(out) == tifs[0].file_size:
            return out
        ensure_disk_free("unzip", extra_gb=tifs[0].file_size / 1024 ** 3)
        try:
            with z.open(tifs[0]) as src, open(out + ".part", "wb") as dst:
                shutil.copyfileobj(src, dst, 16 * 1024 * 1024)       # zipfile raises BadZipFile on a CRC mismatch
        except zipfile.BadZipFile as e:
            os.remove(out + ".part")
            os.remove(zip_path)
            raise PipelineStop(f"{os.path.basename(zip_path)} is damaged ({e}); it was deleted. Run again to fetch it anew.")
        os.replace(out + ".part", out)
    return out


# ---------- Earth Engine's copy (USDA/NASS/CDL) ----------
EE = {}


def ee_ready():
    if not EE:
        import ee
        try:
            ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
        except Exception:
            ee.Authenticate()
            ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
        EE["ee"] = ee
        log.info("Earth Engine ready (for the layers NASS no longer offers)")
    return EE["ee"]


def ee_asset(year):
    """Earth Engine's image of a year, with every band's own grid (they differ in some years)."""
    key = ("asset", year)
    if key not in EE:
        ee = ee_ready()
        try:
            info = ee.data.getAsset(f"projects/earthengine-public/assets/USDA/NASS/CDL/{year}")
        except Exception as e:
            if "not found" in str(e).lower():
                EE[key] = None
                return None
            raise
        EE[key] = info
    return EE[key]


def ee_has(year, layer):
    info = ee_asset(year) if EE_FALLBACK else None
    return bool(info) and any(b["id"] == layer for b in info["bands"])


def class_table(year):
    """NASS's class codes, names and colours of the crop-type layer, as Earth Engine stores them with each year."""
    info = ee_asset(year)
    if not info:
        return None
    p = info.get("properties", {})
    out = {}
    for layer in ("cropland", "cultivated"):
        v, n, c = (p.get(f"{layer}_class_{k}") for k in ("values", "names", "palette"))
        if v is not None and n is not None:
            split = lambda s: s if isinstance(s, list) else str(s).split(",")
            out[layer] = dict(values=[int(x) for x in split(v)], names=[str(x).strip() for x in split(n)],
                              palette=[str(x).strip() for x in split(c)] if c is not None else [])
    return out


def ee_read(year, layer, bounds):
    """The stored values of one band of Earth Engine's copy over `bounds` (x0, y0, x1, y1 in the band's
    own CRS), on the band's own grid: (array, transform, crs). Fetched in pieces Earth Engine accepts."""
    ee = ee_ready()
    info = ee_asset(year)
    band = next(b for b in info["bands"] if b["id"] == layer)
    g = band["grid"]
    at = g["affineTransform"]
    T = Affine(at["scaleX"], at.get("shearX", 0), at["translateX"], at.get("shearY", 0), at["scaleY"], at["translateY"])
    crs = g.get("crsCode") or g.get("crsWkt")
    inv = ~T
    cols, rows = zip(*[inv * (x, y) for x in (bounds[0], bounds[2]) for y in (bounds[1], bounds[3])])
    c0, c1 = max(0, math.floor(min(cols)) - 2), min(g["dimensions"]["width"], math.ceil(max(cols)) + 2)
    r0, r1 = max(0, math.floor(min(rows)) - 2), min(g["dimensions"]["height"], math.ceil(max(rows)) + 2)
    out = np.zeros((max(0, r1 - r0), max(0, c1 - c0)), dtype="u1")
    step = 4000                                          # 4000 x 4000 one-byte values: under Earth Engine's 48 MiB limit

    def piece(a, c):
        h, w = min(step, r1 - a), min(step, c1 - c)
        grid = dict(dimensions=dict(width=w, height=h), crsCode=crs,
                    affineTransform=dict(scaleX=T.a, shearX=0, translateX=T.c + c * T.a, shearY=0, scaleY=T.e, translateY=T.f + a * T.e))
        for attempt in range(5):
            try:
                raw = ee.data.getPixels(dict(assetId=info["name"], bandIds=[layer], grid=grid, fileFormat="NPY"))
                arr = np.load(io.BytesIO(raw), allow_pickle=False)[layer]
                if arr.shape != (h, w) or arr.dtype != np.dtype("u1"):
                    raise IOError(f"unexpected block {arr.dtype} {arr.shape}")
                TOTALS["ee_bytes"] = TOTALS.get("ee_bytes", 0) + len(raw)
                return a, c, arr
            except Exception as e:
                if attempt == 4:
                    raise PipelineStop(f"Earth Engine block {layer} {year} could not be read ({str(e)[:160]}). Run again.")
                time.sleep(5 * 2 ** attempt)

    with ThreadPoolExecutor(8) as ex:
        for a, c, arr in ex.map(lambda rc: piece(*rc), [(a, c) for a in range(r0, r1, step) for c in range(c0, c1, step)]):
            out[a - r0:a - r0 + arr.shape[0], c - c0:c - c0 + arr.shape[1]] = arr
    return out, T * Affine.translation(c0, r0), crs


def sources_of(year, last):
    """Where each layer of a year comes from: {layer: ("nass", zip name) | ("earth_engine", None) | ("none", None)}."""
    out = {}
    for layer in LAYERS:
        name = NASS_FILES[layer].get(year)
        if layer == "cropland" and year == last and name is None:
            name = f"{year}_30m_cdls.zip"
        if name:
            out[layer] = ("nass", name)
        elif ee_has(year, layer):
            out[layer] = ("earth_engine", None)
        else:
            out[layer] = ("none", None)
    return out

In [ ]:
# ═══════════════ CELL 7: place a layer on a tile (nearest neighbour), write the tile-year file, check it ═══════════════
def frame_transform(F):
    return Affine(PIXEL, 0.0, F["x0"], 0.0, -PIXEL, F["y0"])


def source_bounds(F, src_crs):
    """The frame's outline in the source's CRS (its edges sampled densely, since the projections differ)."""
    xs = np.linspace(F["x0"], F["x0"] + F["w"] * PIXEL, 65)
    ys = np.linspace(F["y0"] - F["h"] * PIXEL, F["y0"], 65)
    ex = np.concatenate([xs, xs, np.full(65, xs[0]), np.full(65, xs[-1])])
    ey = np.concatenate([np.full(65, ys[0]), np.full(65, ys[-1]), ys, ys])
    tx, ty = pyproj.Transformer.from_crs(F["crs"], src_crs, always_xy=True).transform(ex, ey)
    return min(tx), min(ty), max(tx), max(ty)


_CENTRES = {}


def centres_in(F, src_crs):
    """The centres of all the frame's pixels in the source's CRS, projected exactly with pyproj (no
    approximation), kept for the tile being worked on."""
    key = (json.dumps(F, sort_keys=True), str(src_crs))
    if key not in _CENTRES:
        _CENTRES.clear()
        tr = pyproj.Transformer.from_crs(F["crs"], src_crs, always_xy=True)
        xs = F["x0"] + (np.arange(F["w"]) + 0.5) * PIXEL
        sx, sy = np.empty((F["h"], F["w"])), np.empty((F["h"], F["w"]))
        for r0 in range(0, F["h"], 512):
            r1 = min(F["h"], r0 + 512)
            y = F["y0"] - (np.arange(r0, r1) + 0.5) * PIXEL
            X, Y = np.meshgrid(xs, y)
            sx[r0:r1], sy[r0:r1] = tr.transform(X, Y)
        _CENTRES[key] = (sx, sy)
    return _CENTRES[key]


def place(src, src_transform, src_crs, F, mask):
    """One layer on the tile's frame: every pixel takes the value of the source pixel that contains the
    pixel's centre (nearest neighbour, computed exactly; no averaging: the value is always one of NASS's).
    Outside the polygon, and where the source does not reach: FILL."""
    if np.count_nonzero(src == FILL):
        raise DataIntegrityError(f"the source holds the value {FILL}, which the files use as their fill value")
    if src_transform.b or src_transform.d:
        raise DataIntegrityError("the source grid is rotated")
    sx, sy = centres_in(F, src_crs)
    col = np.floor((sx - src_transform.c) / src_transform.a).astype(np.int64)
    row = np.floor((sy - src_transform.f) / src_transform.e).astype(np.int64)
    ok = mask & (row >= 0) & (row < src.shape[0]) & (col >= 0) & (col < src.shape[1])
    out = np.full((F["h"], F["w"]), FILL, dtype="u1")
    out[ok] = src[row[ok], col[ok]]
    return out


def read_source(year, layer, how, F):
    """(array, transform, crs) of the part of a layer's source that covers the frame."""
    kind, path = how
    if kind == "nass":
        with rasterio.open(path) as ds:
            b = source_bounds(F, ds.crs)
            win = rasterio.windows.from_bounds(*b, transform=ds.transform)
            c0, r0 = max(0, math.floor(win.col_off) - 2), max(0, math.floor(win.row_off) - 2)
            c1 = min(ds.width, math.ceil(win.col_off + win.width) + 2)
            r1 = min(ds.height, math.ceil(win.row_off + win.height) + 2)
            w = rasterio.windows.Window(c0, r0, c1 - c0, r1 - r0)
            arr = ds.read(1, window=w)
            if arr.dtype != np.dtype("u1"):
                raise DataIntegrityError(f"{os.path.basename(path)} is {arr.dtype}; 8-bit values were expected")
            return arr, ds.window_transform(w), ds.crs
    info = ee_asset(year)
    band = next(b for b in info["bands"] if b["id"] == layer)
    crs = band["grid"].get("crsCode") or band["grid"].get("crsWkt")
    return ee_read(year, layer, source_bounds(F, crs))


def write_tile_year(path, key, F, year, mask, arrays, sources, classes):
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    cy, cx = min(CHUNK_YX, F["h"]), min(CHUNK_YX, F["w"])
    crs = pyproj.CRS.from_user_input(F["crs"])
    tmp = path + ".new"
    with nc4.Dataset(tmp, "w", format="NETCDF4") as ds:
        ds.createDimension("y", F["h"])
        ds.createDimension("x", F["w"])
        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis, yv.long_name = "projection_y_coordinate", "m", "Y", "northing of pixel centre"
        yv[:] = F["y0"] - (np.arange(F["h"]) + 0.5) * PIXEL
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis, xv.long_name = "projection_x_coordinate", "m", "X", "easting of pixel centre"
        xv[:] = F["x0"] + (np.arange(F["w"]) + 0.5) * PIXEL
        gm = ds.createVariable("crs", "i4")
        for k, v in crs.to_cf().items():
            gm.setncattr(k, v if isinstance(v, (str, int, float)) else np.asarray(v) if isinstance(v, (list, tuple)) else str(v))
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{F['x0']} {PIXEL} 0.0 {F['y0']} 0.0 {-PIXEL}"
        gm.long_name = f"the 30 m grid of the Landsat and OpenET files, UTM zone {int(key[1:3])}, tile {key}"
        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), chunksizes=(cy, cx), **zkw)
        mv.long_name = f"1 where the pixel centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values, mv.flag_meanings, mv.grid_mapping = np.array([0, 1], dtype="i1"), "outside_roi inside_roi", "crs"
        mv[:] = mask.astype("i1")
        for layer in LAYERS:
            kind, name = sources[layer]
            v = ds.createVariable(layer, "u1", ("y", "x"), fill_value=np.uint8(FILL), chunksizes=(cy, cx), **zkw)
            v.long_name, v.grid_mapping = LAYER_TEXT[layer], "crs"
            v.units = "percent" if layer == "confidence" else "1"
            v.source = {"nass": f"USDA NASS, {NASS_URL}{name}",
                        "earth_engine": f"Google Earth Engine's copy of the USDA NASS CDL, USDA/NASS/CDL/{year}, band {layer}",
                        "none": "no source offers this layer for this year"}[kind]
            v.resampling = ("nearest neighbour from the source's own Albers grid: each pixel holds the value of the "
                            "source pixel containing its centre")
            if layer == "cultivated" and year >= 2024 and kind == "nass":
                v.comment = "NASS's cultivated layer of this release is built from the five most recent years."
            if layer in classes:
                t = classes[layer]
                v.flag_values = np.array(t["values"], dtype="u1")
                v.flag_meanings = " ".join(re.sub(r"[^0-9A-Za-z]+", "_", n).strip("_") or "unnamed" for n in t["names"])
                v.class_names = json.dumps(dict(zip(t["values"], t["names"])))
                if t["palette"]:
                    v.class_palette = json.dumps(dict(zip(t["values"], t["palette"])))
            s = ds.createVariable(f"{layer}_source", "i1")
            s.long_name = f"where the {layer} layer of this file comes from"
            s.flag_values = np.array(list(SOURCE_CODE.values()), dtype="i1")
            s.flag_meanings = " ".join(SOURCE_CODE)
            s[...] = SOURCE_CODE[kind]
            v[:] = arrays[layer] if layer in arrays else np.full((F["h"], F["w"]), FILL, dtype="u1")
        ds.Conventions = "CF-1.9"
        ds.title = f"USDA NASS Cropland Data Layer {year}, {ROI_LABEL} aquifer, tile {key} of the 30 m Landsat/OpenET grid"
        ds.summary = ("Crop type, confidence and cultivated layers of the CDL placed on the 30 m UTM tiles of this "
                      "project's Landsat and OpenET files by nearest neighbour from NASS's Albers grid. Every value is "
                      "one of NASS's; none is averaged or filled.")
        ds.institution = "USDA National Agricultural Statistics Service (source data); packaged by the MSU groundwater project"
        ds.references = "USDA NASS Cropland Data Layer, https://www.nass.usda.gov/Research_and_Science/Cropland/"
        ds.history = f"{iso(utcnow())} created by USDA_NASS_CDL_30m_Ogallala.ipynb"
        ds.tile, ds.year, ds.roi = key, np.int32(year), ROI_LABEL
        ds.frame = json.dumps(F)
        ds.sources = json.dumps({k: list(v) for k, v in sources.items()})
        ds.processing_status = "complete"
    os.replace(tmp, path)


def check_tile_year(path, F, mask, arrays):
    """Read every layer back from the closed file and compare it with what was computed."""
    with nc4.Dataset(path) as ds:
        ds.set_auto_maskandscale(False)
        if not np.array_equal(np.asarray(ds["roi_mask"][:]).astype(bool), mask):
            raise DataIntegrityError(f"{os.path.basename(path)}: roi_mask differs from what was made")
        for layer, a in arrays.items():
            if not np.array_equal(np.asarray(ds[layer][:]), a):
                raise DataIntegrityError(f"{os.path.basename(path)}: {layer} differs from what was computed")


def spot_check(src, src_transform, src_crs, F, placed, n=2000, seed=0):
    """An independent check of the placement, by another library: for random pixels inside the polygon,
    the pixel centre is projected with GDAL (rasterio.warp.transform, not pyproj) and the source pixel
    holding it is found with rasterio's own index function. Points within a thousandth of a pixel of a
    source pixel edge are skipped (either neighbour is then right)."""
    rr, cc = np.nonzero(placed != FILL)
    if not rr.size:
        return 0
    pick = np.random.default_rng(seed).choice(rr.size, min(n, rr.size), replace=False)
    r, c = rr[pick], cc[pick]
    x, y = F["x0"] + (c + 0.5) * PIXEL, F["y0"] - (r + 0.5) * PIXEL
    sx, sy = rasterio.warp.transform(F["crs"], src_crs, x.tolist(), y.tolist())
    fc, fr = ~src_transform * (np.asarray(sx), np.asarray(sy))
    clear = (np.abs(fc - np.round(fc)) > 1e-3) & (np.abs(fr - np.round(fr)) > 1e-3)
    row, col = rasterio.transform.rowcol(src_transform, np.asarray(sx)[clear], np.asarray(sy)[clear], op=math.floor)
    row, col = np.asarray(row), np.asarray(col)
    bad = np.count_nonzero(src[row, col] != placed[r[clear], c[clear]])
    if bad:
        raise DataIntegrityError(f"{bad} of {int(clear.sum())} sampled pixels differ from a direct lookup in the source")
    return int(clear.sum())

In [ ]:
# ═══════════════ CELL 8: year by year: fetch the sources, every tile, check, Drive; run ═══════════════
# The unit of work is one tile and one year: a small file, built on scratch, checked, and copied to Drive
# under its final name. A year's national files are downloaded once and serve all its tiles. A runtime that
# dies loses the file it was working on; the next run skips every file that is on Drive and complete.
def unit_path(key, year):
    return os.path.join(DRIVE_OUT, str(year), key, f"{PRODUCT_TAG}_{key}_{year}_{ROI_LABEL}.nc")


def done_on_drive(path, sources):
    """True when the file is on Drive, complete, and made from the same sources."""
    if not os.path.exists(path):
        return False
    for attempt in range(3):
        try:
            with nc4.Dataset(path) as ds:
                return (str(getattr(ds, "processing_status", "")) == "complete"
                        and json.loads(str(ds.sources)) == {k: list(v) for k, v in sources.items()})
        except Exception as e:
            why = e
            time.sleep(20 * (attempt + 1))
    raise PipelineStop(f"{os.path.basename(path)} is on Drive but cannot be read ({why}). Nothing was changed. If it "
                       "really is damaged, move it out of the folder yourself and run again.")


def run_year(year, last, frames, prog):
    sources = sources_of(year, last)
    todo = {k: F for k, F in frames.items() if not done_on_drive(unit_path(k, year), sources)}
    log.info("-" * 78)
    log.info("YEAR %d | %s | %d of %d tile files to make", year,
             ", ".join(f"{l}: {s[0]}" for l, s in sources.items()), len(todo), len(frames))
    if not todo:
        return 0
    local = {}
    for layer, (kind, name) in sources.items():
        if kind == "nass":
            local[layer] = ("nass", nass_tif(download_nass(name)))
            if not KEEP_DOWNLOADS:
                os.remove(os.path.join(RAW_DIR, name))
        else:
            local[layer] = (kind, None)
    classes = class_table(year) or {}
    made = 0
    for key, F in sorted(todo.items()):
        t0 = time.time()
        mask = frame_mask(key, F)
        arrays, checked = {}, 0
        for layer, how in local.items():
            if how[0] == "none":
                continue
            src, T, crs = read_source(year, layer, how, F)
            arrays[layer] = place(src, T, crs, F, mask)
            checked += spot_check(src, T, crs, F, arrays[layer], seed=year)
            missing = np.count_nonzero(mask & (arrays[layer] == FILL))
            if missing:
                log.warning("[%s %d] %s: %d aquifer pixels have no value in the source", key, year, layer, missing)
        work = os.path.join(NC_DIR, f"{key}_{year}.nc")
        write_tile_year(work, key, F, year, mask, arrays, sources, classes)
        check_tile_year(work, F, mask, arrays)
        dst = unit_path(key, year)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        robust_drive_copy(work, dst)
        size = os.path.getsize(work)
        os.remove(work)
        log_events(LEDGER_NAME, [dict(file=os.path.basename(dst), tile=key, year=year, size=size,
                                      sources={k: v[0] for k, v in sources.items()})])
        made += 1
        prog[0] += 1
        log.info("[%s %d] DONE: %s on Drive, %d sampled pixels equal to a direct lookup, %.0f s | %d files this run",
                 key, year, human(size), checked, time.time() - t0, prog[0])
        gc.collect()
    if not KEEP_DOWNLOADS:
        for layer, (kind, path) in local.items():
            if kind == "nass" and path and os.path.exists(path):
                os.remove(path)
    return made


def run():
    t0 = time.time()
    TOTALS.clear()
    stopped = None
    try:
        ensure_disk_free("start")
        frames = get_frames()
        if ONLY_TILES:
            frames = {k: v for k, v in frames.items() if k in set(ONLY_TILES)}
        if TEST_MODE:
            keys = sorted(frames, key=lambda k: -frames[k]["n_roi"])
            frames = {k: frames[k] for k in [keys[int((i + 0.5) * len(keys) / TEST_TILES)] for i in range(min(TEST_TILES, len(keys)))]}
        last = nass_latest_year() if not (TEST_MODE or LAST_YEAR) else (TEST_YEAR if TEST_MODE else LAST_YEAR)
        years = [TEST_YEAR] if TEST_MODE else list(range(FIRST_YEAR, last + 1))
        log.info("CDL %d to %d on %d tiles of the 30 m Landsat/OpenET grid -> %s", years[0], years[-1], len(frames), DRIVE_OUT)
        prog = [0]
        for year in years:
            run_year(year, last, frames, prog)
    except PipelineStop as e:
        stopped = str(e)
    except KeyboardInterrupt:
        stopped = "Stopped with the stop button."
    log.info("=" * 78)
    if stopped:
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing on Drive is lost, re-run to resume)\n  {stopped}\n" + "=" * 78 + "\n")
    log.info("  this run: %s downloaded from NASS, %s from Earth Engine, %.1f min", human(TOTALS.get("bytes", 0)),
             human(TOTALS.get("ee_bytes", 0)), (time.time() - t0) / 60)


run()

In [ ]:
# ═══════════════ CELL 9: what is on Drive ═══════════════
def audit():
    rows = {}
    for p in glob.glob(os.path.join(DRIVE_OUT, "[0-9][0-9][0-9][0-9]", "z*", "*.nc")):
        year = int(os.path.basename(os.path.dirname(os.path.dirname(p))))
        r = rows.setdefault(year, dict(files=0, size=0, src={}, bad=0))
        r["files"] += 1
        r["size"] += os.path.getsize(p)
        try:
            with nc4.Dataset(p) as ds:
                for layer in LAYERS:
                    k = int(ds[f"{layer}_source"][...])
                    name = {v: n for n, v in SOURCE_CODE.items()}[k]
                    r["src"].setdefault(layer, set()).add(name)
        except Exception:
            r["bad"] += 1
    print(f"{DRIVE_OUT}\n\n{'year':4s} {'tiles':>5s} {'size':>10s}  sources (cropland | confidence | cultivated)")
    for y in sorted(rows):
        r = rows[y]
        src = " | ".join("/".join(sorted(r["src"].get(l, {"?"}))) for l in LAYERS)
        print(f"{y:4d} {r['files']:5d} {human(r['size']):>10s}  {src}" + (f"   {r['bad']} file(s) unreadable" if r["bad"] else ""))


audit()

# Reading the files:
#   import glob, xarray as xr
#   root  = "/content/drive/MyDrive/MSUGWB/USDA_NASS_CDL_30m_tiles"
#   files = sorted(glob.glob(f"{root}/*/z14e02n29/*.nc"))                 # every year of one tile
#   ds = xr.open_mfdataset(files, combine="nested", concat_dim="year")
#   corn = ds.cropland == 1                                                # class codes: ds.cropland.class_names
# The tiles have the names, x and y of the Landsat and OpenET files, so the three line up pixel for pixel.